# Info from Canvas


## Viterbi Algorithm for Hidden Markov Models
Your group will demonstrate your understanding of Hidden Markov Models by implementing the Viterbi algorithm for finding the most likely sequence of hidden states given a sequence of observations.

The Viterbi algorithm represents a dynamic programming approach to decoding hidden states, offering several advantages over naive approaches:

### Key Features

Optimal Path Finding: Identifies the most probable sequence of hidden states
Dynamic Programming: Uses efficient tabulation to avoid redundant calculations
Log-Space Computation: Prevents numerical underflow with large sequences
Traceback Mechanism: Reconstructs the optimal state path after computation

### Algorithm Structure

The Viterbi algorithm involves these key steps:

1. Initialization:
Set up a probability matrix using initial probabilities and the first observation
Initialize traceback matrix for path reconstruction
2. Recursion:
For each position and possible state, calculate the maximum probability
Store both probabilities and traceback pointers
Apply transition and emission probabilities at each step
3. Termination:
Identify the final state with the highest probability
Trace back through the matrix to reconstruct the optimal path

### Applications Beyond Sequence Analysis

While primarily used for biological sequence analysis, this approach has broader applications:

- Speech Recognition: Identifying phonemes in audio signals
- Part-of-Speech Tagging: Determining grammatical roles in text
- Gene Finding: Locating coding regions in DNA sequences
- Financial Modeling: Detecting market regimes in time series data

### Computational Considerations

Important factors to consider in implementation:

Time Complexity: O(N x K^2) where N is sequence length and K is number of states
Space Complexity: O(N x K) for storing the dynamic programming matrix
Numerical Stability: Using log probabilities to prevent underflow
Edge Cases: Handling zero probabilities with pseudocounts

# Thoughts and considerations

Define HMM class with attributes: 
    - initial probabilities -> Dict 
    - transition probabilities -> Dict of Dicts 
    - emission probabilities -> Dict of Dicts
Track states and potential emissions from given observations

*For viterbi, assumption is all emissions and states are defined... but for future algorithms (potentially) consider ability to add and track emissions/states. 

1) initialization 
    - for each observations, calculate probability that observation occurs in each state given transition probabilities and most recent probability calculated
            - most recent probability * transition probability * emission probability in state.
            - For each state in state tracker, the above calc is done





# Pseudocode

Class HMM(emission_prob, transition_prob, initial_prob):

    FUNCTION viterbi(self, observations):
    
    INITIALIZATION: start with initialization of probabilities for each state for each observation
    initialize two matrices, probability matrix and move matrix 
    first column of probability matrix:
        for each state: 
            initial probability of state k * emission probability for observation 0 in state k

    RECURSION: 
        For each remaining observation: 
            for each possible state:
                previous probability for preceding state  * transition probability * emission probability
                take max value after all states are iterated through
                store state that contributed to max value
                return score matrix, move matrix, max of last column (or Termination function)
                
    TERMINATION:
        after all observations are iterated through
        Take max from last column and begin traceback
    
    TRACEBACK:
        take state contributing to max of last column
        add to state path list 
        move one back in the matrix and look at cell in corresponding state row
        get state contributing to max in new cell
        add to state path list
        move one back in matrix and look at cell in corresponding state row
        repeat for all observations
        reverse list of state path
        return state path list (final output)

    return output from traceback

In [1]:
import numpy as np

In [38]:
class HMM:
    """
    Class HMM implements a Hidden Markov Model with Viterbi decoding.

    Attributes:
        - initial_prob (dict): dictionary of {state: probability} for initial state probabilities
        - transition_prob (dict): dictionary of dictionaries of {state: {state: probability}} for transition probabilities
        - emission_prob (dict): dictionary of dictionaries of {state: {emission: probability}} for emission probabilities
        - states (list): list of state names derived from initial_prob keys
        - emissions (list): list of emission names derived from emission_prob keys
    Methods:
        - _initialization: Initializes the probability and traceback matrices for the Viterbi algorithm
        - _recursion: Fills in the probability and traceback matrices using dynamic programming
        - _traceback: Traces back through the move matrix to recover the optimal path
        - viterbi: Runs the full Viterbi algorithm and returns the most probable state sequence
    """

    def __init__(self, initial_prob, transition_prob, emission_prob):

        self.initial_prob = initial_prob
        self.transition_prob = transition_prob
        self.emission_prob = emission_prob
        self.states = list(initial_prob.keys())
        self.emissions = list(emission_prob[self.states[0]].keys())

    def _initialization(self, observations):
        """
        Initializes the probability and traceback matrices for the Viterbi algorithm.

        Parameters:
            - observations (list): list of observed emissions
        Returns:
            - probability_matrix (np.ndarray): (states x observations) matrix of log probabilities with column 0 filled with log(initial_prob) + log(emission_prob) for each state
            - move_matrix (np.ndarray): (states x observations) traceback matrix with column 0 initialized to each state's own index
        """

        # Initialize probability and traceback matrices
        probability_matrix = np.zeros((self.rows, self.columns))
        move_matrix = np.zeros((self.rows, self.columns))

        # For each possible state, calculate initial_prob * emission 0 at that state
        for state in range(self.rows):
            probability_matrix[state, 0] = np.log(
                self.initial_prob[self.state_index[state]]
            ) + np.log(self.emission_prob[self.state_index[state]][observations[0]])
            move_matrix[state, 0] = state

        return probability_matrix, move_matrix

    def _recursion(self, observations, probability_matrix, move_matrix):
        """
        Fills in the probability and traceback matrices using dynamic programming.

        For each observation and state, computes the maximum log probability over all
        possible previous states using: prev_prob + log(transition_prob) + log(emission_prob).

        Parameters:
            - observations (list): list of observed emissions
            - probability_matrix (np.ndarray): initialized (states x observations) log probability matrix
            - move_matrix (np.ndarray): initialized (states x observations) traceback matrix
        Returns:
            - probability_matrix (np.ndarray): fully filled log probability matrix
            - move_matrix (np.ndarray): fully filled traceback matrix, where each cell stores
              the index of the previous state that yielded the maximum probability
        """

        # Iterate through each observation
        for obs in range(1, self.columns):

            # For each state, find max joint probability
            for state in range(self.rows):

                # Initialize max value trackers
                max_val = -np.inf
                max_obs = 0

                # For each potentially previous state calculate the joint probability of current state
                for prev_state in range(self.rows):
                    prev_prob = probability_matrix[prev_state, obs - 1]
                    trans_prob = np.log(
                        self.transition_prob[self.state_index[prev_state]][
                            self.state_index[state]
                        ]
                    )
                    emission_prob = np.log(
                        self.emission_prob[self.state_index[state]][observations[obs]]
                    )
                    curr_val = prev_prob + trans_prob + emission_prob

                    # If calculated probability > max value --> update
                    if curr_val > max_val:
                        max_val = curr_val
                        max_obs = prev_state

                # Update probability and move matrices with max value and state we transitioned from
                probability_matrix[state, obs] = max_val
                move_matrix[state, obs] = max_obs

        return probability_matrix, move_matrix

    def _traceback(self, move_matrix, best_path_start):
        """
        Traces back through the move matrix to recover the optimal state sequence.

        Starting from the best final state, follows the traceback moves stored in
        move_matrix from right to left to reconstruct the most probable path.

        Parameters:
            - move_matrix (np.ndarray): fully filled (states x observations) traceback matrix
            - best_path_start (int): index of the state with the highest log probability
              in the final column of the probability matrix
        Returns:
            - best_path (list): ordered list of state names representing the most probable
              state sequence for the given observations
        """

        # Create list to track states from best path
        best_state = best_path_start
        best_path = [self.state_index[best_state]]

        # For each column in the matrix, walk back from the last column finding the best state (max)
        for i in reversed(
            range(2, self.columns + 1)
        ):  # Ignore first column as value is initial state
            best_state = int(move_matrix[best_state, i - 1])

            # Append appropriate state name to best path list
            best_path.append(self.state_index[best_state])

        # Return reverse best path
        return best_path[::-1]

    def viterbi(self, observations):
        """
        Runs the full Viterbi algorithm and returns the most probable state sequence.

        Executes initialization, recursion, termination, and traceback steps
        to find the most probable hidden state sequence for a given observation sequence.

        Parameters:
            - observations (list): list of observed emissions
        Returns:
            - best_path (list): ordered list of state names representing the most probable
              state sequence for the given observations
        """

        # Create rows and columns variables
        self.rows = len(self.states)
        self.columns = len(observations)

        # Create index to track what row each state is {0: state1, 1: state2, ...}
        self.state_index = {i: state for i, state in enumerate(self.states)}

        # Initialization step
        probability_matrix, move_matrix = self._initialization(observations)

        # Recursion step
        probability_matrix, move_matrix = self._recursion(
            observations, probability_matrix, move_matrix
        )
        print(probability_matrix)
        print(move_matrix)

        # Termination step --> get row of max value
        best_path_start = int(probability_matrix[:, -1].argmax())

        # Traceback step
        best_path = self._traceback(move_matrix, best_path_start)

        return best_path
    
    
    def forward(self, observations):
        
        self.rows = len(self.states)
        self.columns = len(observations)

        # Create index to track what row each state is {0: state1, 1: state2, ...}
        self.state_index = {i: state for i, state in enumerate(self.states)}



        # Initialization step
        probability_matrix, _ = self._initialization(observations)

        # calculate probability matrix using observations 
        probability_matrix = self._calculate_matrix(probability_matrix, observations)

        # calculate overall probability (sum of last column)
        overall_probability = np.logaddexp.reduce(probability_matrix[:, -1])

        return probability_matrix, overall_probability

        # Create rows and columns variables
        # Create index to track what row each state is {0: state1, 1: state2, ...}
        # Initialization step (same function as Viterbi, don't track move matrix in this case)
        # Fill in calculation matrix 
        # Calculate overall probability (sum of last column)
        # Return matrix and probability
    
    def _calculate_matrix(self, probability_matrix, observations):


        # Iterate through each observation
        for obs in range(1, self.columns):

            # For each state, find max joint probability
            for state in range(self.rows):

                # Initialize cumulative probabilty
                cumulative_prob = -np.inf

                # For each potentially previous state calculate the joint probability of current state
                for prev_state in range(self.rows):
                    prev_prob = probability_matrix[prev_state, obs - 1]
                    trans_prob = np.log(
                        self.transition_prob[self.state_index[prev_state]][
                            self.state_index[state]
                        ]
                    )
                    emission_prob = np.log(
                        self.emission_prob[self.state_index[state]][observations[obs]]
                    )
                    curr_val = prev_prob + trans_prob + emission_prob

                    # If calculated probability > max value --> update
                    cumulative_prob = float(np.logaddexp(cumulative_prob, curr_val))

                # Update probability and move matrices with max value and state we transitioned from
                probability_matrix[state, obs] = cumulative_prob

        return probability_matrix
        # Iterate through each observation
            # For each state, sum joint probabilities
                # Initialize cumulative probabilty
                # For each potentially previous state calculate the joint probability of current state
                    # Add to cumulative probability
                # Update probability matrix with cumulative probability
    
    def backward(self, observations):
        
        # Reverse Observations
        reverse_obs = observations[::-1]

        # Run foward algorithm on reversed observations
        backwards_prob_matrix, backwards_overall_probability = self.forward(reverse_obs)


        return backwards_prob_matrix, backwards_overall_probability
        
        

    
    def forward_backward(self, observations, index):
        
        # Call forward algorithm
        forward_matrix, forward_probability = self.forward(observations)

        # Call backward algorithm
        backwards_matrix, backwards_probability = self.backward(observations)

        # Calculate average overall probability
        average_probability = (forward_probability + backwards_probability) / 2

        # Access forward matrix at position index --> fk
        marg_probs = {}

        for row, state in self.state_index.items():
            fk = forward_matrix[row, index]
            # Access backward matrix at position (len(observations) - index) --> bk
            bk = backwards_matrix[row, len(observations) - index - 1]
            # Calculate marginal posterior probability (fk * bk / overall prob)
            marginal_posterior = fk + bk - average_probability
            marg_probs[state] = marginal_posterior
    

        return marg_probs

cumulative prob : 0.6931471805599453 + ln(i): 0.6931471805599453 
cumulative prob : 2.089664518466995 + ln(i): 0.6931471805599453 
cumulative prob : 3.098974612332721 + ln(i): 1.0986122886681098 
cumulative prob : 4.0552073394383354 + ln(i): 1.3862943611198906 
cumulative prob : 5.026957141363116 + ln(i): 1.6094379124341003 


In [3]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {"I": 0.2, "G": 0.8}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {"I": {"I": 0.7, "G": 0.3}, "G": {"I": 0.1, "G": 0.9}}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
}

In [5]:
hmm = HMM(init_probs, trans_probs, emit_probs)
#hmm.viterbi(obs)
hmm.forward(obs)

(array([[ -2.52572864,  -3.54737989,  -4.66619489,  -7.21384575,
          -8.03470778, -10.46819339, -11.08566768, -13.4635246 ,
         -15.38816579],
        [ -1.83258146,  -3.39322921,  -4.85671321,  -5.8272255 ,
          -7.46200628,  -8.59906846, -10.2637257 , -11.43632276,
         -12.70269125]]),
 np.float64(-12.6367267220508))

In [11]:
# Example observation sequence
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {"I": 0.1, "G": 0.9}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {"I": {"I": 0.6, "G": 0.4}, "G": {"I": 0.1, "G": 0.9}}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4},
}

In [39]:
hmm2 = HMM(init_probs, trans_probs, emit_probs)
#hmm.viterbi(obs)
#hmm2.forward(obs)

#hmm2.backward(obs)
hmm2.forward_backward(obs, 3)

{'I': np.float64(-2.8017891169975453), 'G': np.float64(-4.195216905339594)}